# Library Database — Synthetic Data + MySQL Load

This notebook generates synthetic data for the four main tables in our library database:

- `Book`
- `Book_copy`
- `Member`
- `Loan`

It also saves CSV backups and loads the data directly into MySQL.

## 1. Imports and settings

Install the packages once if needed:

```bash
pip install pandas faker mysql-connector-python
```

In [1]:
import pandas as pd
import random
import mysql.connector

from faker import Faker
from datetime import date, timedelta
from pathlib import Path
from bisect import bisect_right
from getpass import getpass

fake = Faker()
Faker.seed(42)
rng = random.Random(42)

N_BOOKS = 1000
N_COPIES = 2000
N_MEMBERS = 5000
N_LOANS = 20000

START_DATE = date(2024, 1, 1)
AS_OF_DATE = date(2026, 9, 1)

## 2. Generate `Book`

In [2]:
genres = [
    "Fiction", "Mystery", "Science Fiction", "Fantasy", "Biography",
    "History", "Science", "Religion", "Romance", "Self-Help"
]

books = []
for book_id in range(1, N_BOOKS + 1):
    books.append({
        "book_id": book_id,
        "title": fake.sentence(nb_words=4).rstrip(".")[:45],
        "author": fake.name()[:45],
        "genre": rng.choice(genres),
        "publication_date": fake.date_between(date(1950, 1, 1), date(2025, 12, 31))
    })

df_books = pd.DataFrame(books)
df_books.head()

,book_id,title,author,genre,publication_date
0,1,Agent every,Noah Rhodes,Mystery,2017-10-21
1,2,Opportunity all,David Guzman,Fiction,1992-08-27
2,3,Information last everything thank serve,Jay Ramirez,Biography,1966-05-13
3,4,Bill here grow gas,Andrew Stevens,Fantasy,1959-06-28
4,5,Bad fall pick those,Corey Jones,Fantasy,2000-04-03


## 3. Generate `Book_copy`

Each physical copy belongs to one book. A copy cannot be acquired before its publication date.

In [3]:
conditions = ["New", "Good", "Fair", "Poor"]
copies = []

for copy_id in range(1, N_COPIES + 1):
    book_id = ((copy_id - 1) % N_BOOKS) + 1
    pub_date = df_books.loc[df_books.book_id == book_id, "publication_date"].iloc[0]
    acquisition_start = max(pub_date, date(2015, 1, 1))

    copies.append({
        "copy_id": copy_id,
        "acquisition_date": fake.date_between(acquisition_start, AS_OF_DATE),
        "condition": rng.choice(conditions),
        "status": "Available",
        "Book_book_id": book_id
    })

df_copies = pd.DataFrame(copies)
df_copies.head()

,copy_id,acquisition_date,condition,status,Book_book_id
0,1,2020-02-06,Fair,Available,1
1,2,2023-01-04,Fair,Available,2
2,3,2023-04-28,Poor,Available,3
3,4,2022-09-02,Good,Available,4
4,5,2016-08-30,Poor,Available,5


## 4. Generate `Member`

In [4]:
members = []

for member_id in range(1, N_MEMBERS + 1):
    members.append({
        "member_id": member_id,
        "first_name": fake.first_name()[:45],
        "last_name": fake.last_name()[:45],
        "phone_num": fake.numerify("###-###-####"),
        "member_date": fake.date_between(date(2020, 1, 1), AS_OF_DATE)
    })

df_members = pd.DataFrame(members)
df_members.head()

,member_id,first_name,last_name,phone_num,member_date
0,1,Shawn,Hardin,176-741-2415,2025-05-10
1,2,Julie,Walker,152-854-7815,2025-12-12
2,3,Benjamin,Garcia,853-800-3296,2025-10-24
3,4,Jeffrey,Gonzalez,727-274-2318,2025-09-18
4,5,Kara,Smith,178-932-0649,2022-07-15


## 5. Generate `Loan`

The loan logic keeps the data realistic:

- A member must have joined before borrowing.
- A copy must be available before it can be borrowed.
- Due dates are 14 days after checkout.
- Some recent loans remain checked out.

In [5]:
member_records = sorted(zip(df_members.member_date, df_members.member_id))
member_dates = [x[0] for x in member_records]
member_ids = [x[1] for x in member_records]

next_available = dict(zip(df_copies.copy_id, df_copies.acquisition_date))
total_days = (AS_OF_DATE - START_DATE).days
checkout_dates = sorted(
    START_DATE + timedelta(days=rng.randint(0, total_days))
    for _ in range(N_LOANS)
)

loans = []

for loan_id, date_out in enumerate(checkout_dates, start=1):
    available = [c for c, d in next_available.items() if d <= date_out]
    if not available:
        raise ValueError(f"No copies available on {date_out}")

    copy_id = rng.choice(available)
    eligible_count = bisect_right(member_dates, date_out)
    member_id = rng.choice(member_ids[:eligible_count])

    due_date = date_out + timedelta(days=14)
    return_date = date_out + timedelta(days=rng.randint(3, 21))

    if date_out >= AS_OF_DATE - timedelta(days=30) and rng.random() < 0.35:
        return_date = None
    elif return_date > AS_OF_DATE:
        return_date = None

    if return_date is not None:
        status = "Returned"
        next_available[copy_id] = return_date + timedelta(days=1)
    else:
        status = "Overdue" if due_date < AS_OF_DATE else "Checked Out"
        next_available[copy_id] = date.max

    loans.append({
        "loan_id": loan_id,
        "date_out": date_out,
        "due_date": due_date,
        "status": status,
        "return_date": return_date,
        "Book_copy_copy_id": copy_id,
        "Member_member_id": member_id
    })

df_loans = pd.DataFrame(loans)

checked_out = set(df_loans.loc[df_loans.return_date.isna(), "Book_copy_copy_id"])
df_copies["status"] = df_copies.copy_id.apply(
    lambda x: "Checked Out" if x in checked_out else "Available"
)

df_loans.head()

,loan_id,date_out,due_date,status,return_date,Book_copy_copy_id,Member_member_id
0,1,2024-01-01,2024-01-15,Returned,2024-01-07,942,656
1,2,2024-01-01,2024-01-15,Returned,2024-01-06,1939,1060
2,3,2024-01-01,2024-01-15,Returned,2024-01-19,554,12
3,4,2024-01-01,2024-01-15,Returned,2024-01-16,698,754
4,5,2024-01-01,2024-01-15,Returned,2024-01-16,1848,3620


## 6. Quick validation

These checks should all return `0`.

In [6]:
print("Invalid Book IDs:", (~df_copies.Book_book_id.isin(df_books.book_id)).sum())
print("Invalid Copy IDs:", (~df_loans.Book_copy_copy_id.isin(df_copies.copy_id)).sum())
print("Invalid Member IDs:", (~df_loans.Member_member_id.isin(df_members.member_id)).sum())

member_check = df_loans.merge(
    df_members[["member_id", "member_date"]],
    left_on="Member_member_id",
    right_on="member_id"
)
print("Loans before membership:", (member_check.date_out < member_check.member_date).sum())

Invalid Book IDs: 0
Invalid Copy IDs: 0
Invalid Member IDs: 0
Loans before membership: 0


## 7. Save CSV backups

In [7]:
output_dir = Path.cwd() / "Library_Synthetic_Data"
output_dir.mkdir(exist_ok=True)

df_books.to_csv(output_dir / "Book.csv", index=False)
df_copies.to_csv(output_dir / "Book_copy.csv", index=False)
df_members.to_csv(output_dir / "Member.csv", index=False)
df_loans.to_csv(output_dir / "Loan.csv", index=False)

print(output_dir)

/Users/joshtsamuel/Downloads/Library_Synthetic_Data


## 8. Connect to MySQL

Enter the same MySQL password you use in Workbench.

In [8]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("MySQL password: "),
    database="library"
)

cursor = connection.cursor()
print("Connected to MySQL")

Connected to MySQL


## 9. Reset and load the four tables

This clears only `Loan`, `Book_copy`, `Member`, and `Book`. It does not touch the log tables.

In [9]:
for table in ["Loan", "Book_copy", "Member", "Book"]:
    cursor.execute(f"DELETE FROM `{table}`")

book_sql = """
INSERT INTO Book (book_id, title, author, genre, publication_date)
VALUES (%s, %s, %s, %s, %s)
"""

copy_sql = """
INSERT INTO Book_copy (copy_id, acquisition_date, `condition`, status, Book_book_id)
VALUES (%s, %s, %s, %s, %s)
"""

member_sql = """
INSERT INTO Member (member_id, first_name, last_name, phone_num, member_date)
VALUES (%s, %s, %s, %s, %s)
"""

loan_sql = """
INSERT INTO Loan
(loan_id, date_out, due_date, status, return_date, Book_copy_copy_id, Member_member_id)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(book_sql, list(df_books.itertuples(index=False, name=None)))
cursor.executemany(copy_sql, list(df_copies.itertuples(index=False, name=None)))
cursor.executemany(member_sql, list(df_members.itertuples(index=False, name=None)))

loan_rows = [
    (
        int(r.loan_id), r.date_out, r.due_date, r.status,
        None if pd.isna(r.return_date) else r.return_date,
        int(r.Book_copy_copy_id), int(r.Member_member_id)
    )
    for r in df_loans.itertuples(index=False)
]

cursor.executemany(loan_sql, loan_rows)
connection.commit()

print("Data loaded successfully")

Data loaded successfully


## 10. Verify row counts

In [10]:
for table in ["Book", "Book_copy", "Member", "Loan"]:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}`")
    print(f"{table}: {cursor.fetchone()[0]:,}")

cursor.close()
connection.close()

Book: 1,000
Book_copy: 2,000
Member: 5,000
Loan: 20,000
